# Fig. 10 — effective edge tension over time (local and global, $D$ = 0.1 and 0.5)

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
import numpy as np
import pyprojroot.here as here
import pandas as pd
from vertex_lite.custom import load_data_v3 as load_data
import gc

In [ ]:
def filter_data(local=False, cluster=False):
    if cluster:
        rpath = here('results/resultados_cluster')
    else:
        rpath = here('results/resultados_random')

    dfs = []
    for n in [0.1, 0.5]:
         dfs.append(load_data(
             f'{rpath}/noise_{n}/pct_*/sim_*/cells.parquet',
             columns=['time','cell_id', 'neighbor_of_mutant', 'alive','mutant','t_eff']
         ))
    
    df = pd.concat(dfs, ignore_index=True)
    if df.empty:
        print("No data found")
        return pd.DataFrame()

    df = df[df['alive'] == 1]

    if local:
        df = df[df['neighbor_of_mutant'] == True]
    else:
        df = df.drop(columns=['neighbor_of_mutant'])
    
    return df

In [ ]:
def calculate_teff_stats(df):
    """Mean and SEM of T_eff over all selected cells at each (time, D, f_mut)."""
    groupby_cols = ['time', 'mesh_type', 'pct_mutant']
    teff_stats = (
        df.groupby(groupby_cols)['t_eff']
        .agg(teff_global='mean', teff_global_sem='sem')
        .reset_index()
    )
    return teff_stats

In [ ]:
def plot_teff(df, noise, toplot: str = None, local=False, cluster=False, max_time=None, fig=None, ax=None):
    """
    Plot Teff vs time for every mutant fraction at a given D, supporting overlaying.
    """
    if toplot is None:
        return None, None

    toplot_sem = f"{toplot}_sem"

    sns.set_theme(
        style="white",
        context="paper",
        rc={
            "axes.edgecolor": "black",
            "axes.linewidth": 1.0,
            "axes.facecolor": "white",
            "figure.facecolor": "white",
        }
    )

    d0 = df[df["mesh_type"] == noise].copy()
    if d0.empty:
        return fig, ax

    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(6.5, 4))

    pcts = sorted(d0["pct_mutant"].unique())
    norm = colors.Normalize(vmin=min(pcts), vmax=max(pcts))
    if cluster:
        cmap = sns.light_palette("#A53379", as_cmap=True)
        label_suffix = " (cluster)"
    else:
        cmap = sns.light_palette("#709E0D", as_cmap=True)
        label_suffix = " (random)"

    for p in pcts:
        d = d0[d0["pct_mutant"] == p].sort_values("time")

        if max_time is not None:
            d = d[d["time"] <= max_time]

        x = d["time"].to_numpy()

        y = d[toplot].to_numpy()
        sem = d[toplot_sem].to_numpy()
        y_pos = np.clip(y, 1e-12, None)
        low = np.maximum(y_pos - np.abs(sem), 1e-12)
        high = np.maximum(y_pos + np.abs(sem), 1e-12)
        color = cmap(norm(p))

        ax.fill_between(x, low, high, color=color, alpha=0.12, linewidth=0)
        ax.plot(x, y_pos, color=color, linewidth=1.5, linestyle="-", label=f"mesh {p}{label_suffix}")

    ax.set_ylim(bottom=0.25, top=0.5)
    ax.set_title(f"{toplot} | Noise {noise}", fontsize=14)
    ax.set_xlabel("Time step", fontsize=16)
    ax.set_ylabel(f"Teff {'local' if local else 'global'}", fontsize=16)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(
        axis='both', which='both', direction='out',
        length=5, labelsize=16, width=1, bottom=True, left=True
    )
    ax.tick_params(axis='x', rotation=45)

    return fig, ax

In [ ]:
save_dir = here("results/analysis/teff/plots")
os.makedirs(save_dir, exist_ok=True)
toplot = "teff_global"  # mean over all selected cells (local set or whole tissue)
maxtime = 5000
meshes = [0.1, 0.5]

for local in [True, False]:
    df_random = filter_data(local=local, cluster=False)
    df_cluster = filter_data(local=local, cluster=True)
    stats_random = calculate_teff_stats(df_random)
    stats_cluster = calculate_teff_stats(df_cluster)
    del df_random, df_cluster

    for m in meshes:
        # 1. Random data as the base layer
        fig, ax = plot_teff(
            stats_random, noise=m, toplot=toplot,
            local=local, cluster=False, max_time=maxtime
        )

        # 2. Cluster data overlaid on the same axis
        fig, ax = plot_teff(
            stats_cluster, noise=m, toplot=toplot,
            local=local, cluster=True, max_time=maxtime,
            fig=fig, ax=ax
        )

        if fig is not None:
            plt.tight_layout()
            local_str = 'local' if local else 'global'
            fig.savefig(f"{save_dir}/{toplot}_noise_{m}_{local_str}_overlay_cluster_vs_random_linear.svg", format="svg")
            plt.show()
            plt.close(fig)

    del stats_random, stats_cluster
    gc.collect()